In [ ]:
# =============================================================================
# CycleLayer MVP — Google Colab Training Script
# N-CMAPSS turbofan RUL prediction (physics-informed Brayton cycle layer)
#
# Compatible with: Colab (T4/A100/V100), Python 3.11+, PyTorch >= 2.2
# Usage: paste cells into a Colab notebook, or run as a script.
# =============================================================================

## Cell 0 — Quick-start checklist
1. Runtime > Change runtime type > **GPU** (T4 is fine)
2. Upload your HDF5 file to Google Drive, note its path
3. Edit the USER CONFIG block in Cell 2 (repo URL, HDF5 path, model type)
4. Run all cells in order

--- Cell: 1 — GPU / Environment Check ---

In [ ]:
import subprocess, sys, os, shutil, time, json, textwrap
from pathlib import Path
from datetime import datetime

def _sh(cmd, check=True, **kw):
    """Run shell command, stream output, raise on failure."""
    print(f"\n$ {cmd}")
    proc = subprocess.run(cmd, shell=True, check=check, **kw)
    return proc

# GPU check
try:
    _sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", check=False)
except Exception:
    print("[WARN] nvidia-smi failed or no GPU found.")
import torch
print(f"\ntorch version : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("[WARN] No GPU detected — training will be slow on CPU.")


# --- Cell: 2 — USER CONFIG (edit before running) ---

In [ ]:

# --------------------------------------------------------------------------
# Repo setup — choose ONE of the options below (A or B).
# Option A: Clone from GitHub (first run or if repo not yet mounted)
# Option B: Repo already available (mounted drive or previously cloned)
# --------------------------------------------------------------------------
REPO_URL   = "https://github.com/RobertKunte/cyclelayer-mvp.git"  # ← EDIT
REPO_CLONE = True          # Set False if repo already exists at REPO_ROOT

REPO_ROOT  = Path("/content/cyclelayer-mvp")   # where repo lives in Colab

# --------------------------------------------------------------------------
# Dataset path — HDF5 on Google Drive
# --------------------------------------------------------------------------
DRIVE_DATA_PATH = Path("/content/drive/MyDrive/cyclelayer-mvp/data/NCMAPSS/N-CMAPSS_DS01-005.h5")  # ← EDIT

# --------------------------------------------------------------------------
# Training settings  (shared by both CNN baseline and CycleLayer)
# --------------------------------------------------------------------------
STRIDE_TRAIN       = 5     # 5 = dense (recommended); 10 = faster; 50 = quick test
STRIDE_EVAL        = 1     # keep at 1 for correct PH computation
EPOCHS             = 50
PATIENCE           = 15
BATCH_SIZE         = 512   # reduce to 256 if OOM
OOM_FALLBACK_BATCH = 256   # used automatically on CUDA OOM
NUM_WORKERS        = 4     # Colab typically has 4 CPU cores
SEED               = 42

# --------------------------------------------------------------------------
# LOUO cross-validation (Cell 9 — Advanced / Optional)
# Runs 6 leave-one-unit-out folds for the CycleLayer model (~6× wall-time).
# --------------------------------------------------------------------------
RUN_LOUO = False           # set True to enable

# --------------------------------------------------------------------------
# Artifact destinations
# --------------------------------------------------------------------------
RUNS_ROOT       = Path("/content/runs")
DRIVE_RUNS_ROOT = Path("/content/drive/MyDrive/cyclelayer_runs")  # ← EDIT (or None)

print("Config:")
for k, v in dict(
    STRIDE_TRAIN=STRIDE_TRAIN, STRIDE_EVAL=STRIDE_EVAL,
    EPOCHS=EPOCHS, PATIENCE=PATIENCE,
    BATCH_SIZE=BATCH_SIZE, NUM_WORKERS=NUM_WORKERS,
    SEED=SEED, RUN_LOUO=RUN_LOUO,
).items():
    print(f"  {k:22s} = {v}")


In [ ]:
_sh("pip install -q -U pip")
_sh(
    "pip install -q "
    "torch>=2.2 "
    "numpy>=1.26 "
    "scipy>=1.12 "
    "h5py>=3.10 "
    "matplotlib>=3.8 "
    "pandas>=2.2 "
    "tensorboard>=2.16 "
    "pyyaml>=6.0 "
    "tqdm>=4.66"
)
print("Dependencies installed.")


# --- Cell: 4 — Clone / Load Repo ---

In [ ]:
import shutil
if REPO_CLONE:
    if REPO_ROOT.exists():
        print(f"Repo already exists at {REPO_ROOT}. Pulling latest...")
        try:
            _sh(f"git -C {REPO_ROOT} pull --ff-only")
        except Exception:
            print("[WARN] Pull failed. Force re-cloning...")
            shutil.rmtree(REPO_ROOT, ignore_errors=True)
            _sh(f"git clone {REPO_URL} {REPO_ROOT}")
    else:
        shutil.rmtree(REPO_ROOT, ignore_errors=True)
        _sh(f"git clone {REPO_URL} {REPO_ROOT}")
else:
    assert REPO_ROOT.exists(), f"REPO_ROOT not found: {REPO_ROOT}"
    print(f"Using existing repo at {REPO_ROOT}")

# Make sure repo is importable
for p in [str(REPO_ROOT / "src"), str(REPO_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Verify key entry points exist
for script in ["scripts/train.py", "scripts/evaluate.py", "scripts/run_louo.py"]:
    assert (REPO_ROOT / script).exists(), f"Missing: {script}"

print(f"\nGit commit: ", end="")
_sh(f"git -C {REPO_ROOT} rev-parse --short HEAD")

# ---------------------------------------------------------------------------
# Recover src/cyclelayer/data/ if it was excluded by .gitignore
#
# Background: .gitignore contains the pattern "data/" which accidentally
# matches src/cyclelayer/data/ (the Python package) in addition to the
# intended ./data/ directory (large HDF5 files).
#
# Recovery strategy (in order of preference):
#   1. Files are already present (git was fixed or repo was not affected)
#   2. Copy from Google Drive  (upload src/cyclelayer/data/ there once)
#   3. Raise a clear error with instructions
# ---------------------------------------------------------------------------
_data_pkg = REPO_ROOT / "src" / "cyclelayer" / "data"
_data_pkg_ok = (
    _data_pkg.exists()
    and (_data_pkg / "__init__.py").exists()
    and (_data_pkg / "ncmapss.py").exists()
    and (_data_pkg / "splits.py").exists()
    and (_data_pkg / "preprocessing.py").exists()
)

if not _data_pkg_ok:
    print("\n[WARN] src/cyclelayer/data/ is incomplete in the cloned repo.")
    print("       This is caused by .gitignore 'data/' matching the Python package.")

    # Try to restore from Google Drive
    _drive_src = Path("/content/drive/MyDrive/cyclelayer-mvp/src/cyclelayer/data")
    if _drive_src.exists() and (_drive_src / "ncmapss.py").exists():
        print(f"       Restoring from Drive: {_drive_src} ...")
        _data_pkg.mkdir(parents=True, exist_ok=True)
        for _f in _drive_src.glob("*.py"):
            shutil.copy2(_f, _data_pkg / _f.name)
        print(f"       Restored: {[f.name for f in _data_pkg.glob('*.py')]}")
    else:
        raise RuntimeError(
            "\n"
            "src/cyclelayer/data/ is missing from the cloned repo and from Drive.\n"
            "\n"
            "PERMANENT FIX (recommended):\n"
            "  In .gitignore, change the line:\n"
            "      data/\n"
            "  to:\n"
            "      /data/\n"
            "  This anchors the rule to the repo root, so the HDF5 files in ./data/\n"
            "  remain excluded, but src/cyclelayer/data/*.py (Python source, ~20 KB)\n"
            "  can be committed. Then: git add src/cyclelayer/data/ && git push\n"
            "\n"
            "QUICK WORKAROUND (no git changes required):\n"
            f"  Upload the folder src/cyclelayer/data/ from your local machine to:\n"
            f"  Google Drive -> cyclelayer-mvp/src/cyclelayer/data/\n"
            f"  (copy the 4 .py files there, then re-run this cell)\n"
        )
else:
    print("src/cyclelayer/data/ OK:", [f.name for f in _data_pkg.glob("*.py")])

# Install package so subprocess Python can also import it
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    cwd=str(REPO_ROOT), check=True,
)
print("Package installed (editable).")


In [ ]:
from google.colab import drive  # noqa: E402  (Colab-specific)
drive.mount("/content/drive", force_remount=False)

assert DRIVE_DATA_PATH.exists(), (
    f"\nHDF5 file not found: {DRIVE_DATA_PATH}\n"
    "Ensure the file is uploaded to Google Drive at the path set in USER CONFIG."
)

file_size_gb = DRIVE_DATA_PATH.stat().st_size / 1e9
print(f"HDF5 found: {DRIVE_DATA_PATH}  ({file_size_gb:.2f} GB)")

# Quick sanity check — list top-level keys
import h5py
with h5py.File(DRIVE_DATA_PATH, "r") as f:
    keys = list(f.keys())
    print(f"HDF5 keys : {keys}")
    assert any("dev" in k for k in keys), (
        f"Expected keys ending in '_dev'. Found: {keys}"
    )
print("HDF5 structure OK.")


# --- Cell: 6 — Build Patched Config ---

In [ ]:

# --- Cell: 5b — HDF5 Inspector (optional but recommended) ---
# Prints dataset shapes, dtypes, unit IDs, and the key naming convention.
# Runs fast — no full array loads.

_sh(
    f"{sys.executable} scripts/inspect_hdf5.py "
    f"{DRIVE_DATA_PATH} --units",
    cwd=REPO_ROOT,
)


In [ ]:

# --- Cell: 6 — Build Configs (CycleLayer + CNN baseline) ---
import yaml, torch
from datetime import datetime

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

RUNTIME_CFG_DIR = Path("/content/configs_runtime")
RUNTIME_CFG_DIR.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)


def _apply_shared_overrides(cfg: dict, run_dir: Path) -> None:
    """Patch data / training / evaluation sections in-place with Colab values."""
    cfg.setdefault("data", {})
    cfg["data"]["hdf5_path"]    = str(DRIVE_DATA_PATH)
    cfg["data"]["stride_train"] = STRIDE_TRAIN
    cfg["data"]["stride_eval"]  = STRIDE_EVAL
    cfg["data"]["batch_size"]   = BATCH_SIZE
    cfg["data"]["num_workers"]  = NUM_WORKERS
    cfg.setdefault("training", {})
    cfg["training"]["epochs"]                  = EPOCHS
    cfg["training"]["early_stopping_patience"] = PATIENCE
    cfg["training"]["output_dir"]              = str(run_dir)
    cfg["training"]["use_amp"]                 = torch.cuda.is_available()
    cfg.setdefault("evaluation", {})
    cfg["evaluation"]["checkpoint"]            = str(run_dir / "best.pt")
    cfg["seed"] = SEED


# ── CycleLayer config ─────────────────────────────────────────────────────
# No split_dir override — train.py creates deterministic splits on first run
# and writes them to REPO_ROOT/splits/<dataset_stem>/
RUN_DIR_CL = RUNS_ROOT / f"{RUN_ID}_cyclelayer_v1"
RUN_DIR_CL.mkdir(parents=True, exist_ok=True)

with open(REPO_ROOT / "configs" / "cyclelayer.yaml") as _f:
    cfg_cl = yaml.safe_load(_f)
_apply_shared_overrides(cfg_cl, RUN_DIR_CL)

PATCHED_CFG_CL = RUNTIME_CFG_DIR / f"cyclelayer_v1_stride{STRIDE_TRAIN}.yaml"
with open(PATCHED_CFG_CL, "w") as _f:
    yaml.dump(cfg_cl, _f, default_flow_style=False, sort_keys=False)


# ── CNN baseline config ───────────────────────────────────────────────────
# Pins split_dir to reuse the splits written by CycleLayer training above.
# (Both models see the same train/val/test engine units.)
_splits_dir = str(REPO_ROOT / "splits" / DRIVE_DATA_PATH.stem)

RUN_DIR_CNN = RUNS_ROOT / f"{RUN_ID}_cnn"
RUN_DIR_CNN.mkdir(parents=True, exist_ok=True)

with open(REPO_ROOT / "configs" / "baseline.yaml") as _f:
    cfg_cnn = yaml.safe_load(_f)
_apply_shared_overrides(cfg_cnn, RUN_DIR_CNN)
cfg_cnn["data"]["split_dir"] = _splits_dir   # reuse CycleLayer splits

PATCHED_CFG_CNN = RUNTIME_CFG_DIR / f"cnn_stride{STRIDE_TRAIN}.yaml"
with open(PATCHED_CFG_CNN, "w") as _f:
    yaml.dump(cfg_cnn, _f, default_flow_style=False, sort_keys=False)


# ── Comparison output dir ─────────────────────────────────────────────────
COMPARISON_DIR = RUNS_ROOT / f"{RUN_ID}_comparison"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

print(f"CycleLayer config : {PATCHED_CFG_CL}")
print(f"CycleLayer run dir: {RUN_DIR_CL}")
print(f"CNN config        : {PATCHED_CFG_CNN}")
print(f"CNN run dir       : {RUN_DIR_CNN}")
print(f"Comparison dir    : {COMPARISON_DIR}")
print(f"\nShared splits dir : {_splits_dir}  (written by CycleLayer, reused by CNN)")
print(f"use_amp           : {torch.cuda.is_available()}")


In [ ]:

# --- Cell: 7 — Train CycleLayer + CNN baseline ---

def _stream_subprocess(
    cmd: list,
    logfile: Path,
    cwd: Path,
    cfg_dict: dict | None = None,   # pass to enable OOM-fallback patching
    cfg_path: Path | None = None,   # config file to rewrite on OOM
) -> None:
    """Run subprocess, print output live, tee to logfile.

    OOM fallback: if CUDA OOM is detected and cfg_dict + cfg_path are provided,
    batch_size is patched to OOM_FALLBACK_BATCH and the command is retried once.
    """
    logfile.parent.mkdir(parents=True, exist_ok=True)
    cmd = [str(c) for c in cmd]
    print(f"Running: {' '.join(cmd)}\nLog: {logfile}\n{'='*70}")
    t0 = time.time()

    with open(logfile, "w", encoding="utf-8") as _log:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            cwd=str(cwd),
            env=dict(os.environ,
                     PYTHONPATH=f"{cwd}/src:{os.environ.get('PYTHONPATH', '')}"),
        )
        for line in proc.stdout:
            print(line, end="", flush=True)
            _log.write(line)
        proc.wait()

    elapsed = time.time() - t0
    print(f"\n{'='*70}")
    print(f"Elapsed: {elapsed/60:.1f} min  |  Return code: {proc.returncode}")

    if proc.returncode != 0:
        log_text = open(logfile, encoding="utf-8").read()
        if "CUDA out of memory" in log_text and cfg_dict is not None and cfg_path is not None:
            print(f"\n[OOM] Retrying with batch_size={OOM_FALLBACK_BATCH} ...")
            cfg_dict["data"]["batch_size"] = OOM_FALLBACK_BATCH
            with open(cfg_path, "w") as _f:
                yaml.dump(cfg_dict, _f, default_flow_style=False, sort_keys=False)
            _stream_subprocess(cmd, logfile.with_suffix(".retry.log"), cwd)
        else:
            raise RuntimeError(f"Command failed (code {proc.returncode}). See {logfile}")


# ── Train CycleLayer (runs first so unit splits are written to disk) ──────
LOG_DIR_CL = RUN_DIR_CL / "logs"
LOG_DIR_CL.mkdir(exist_ok=True)

_stream_subprocess(
    [sys.executable, "scripts/train.py",
     "--config", PATCHED_CFG_CL, "--model", "cyclelayer_v1"],
    LOG_DIR_CL / "train.log",
    cwd=REPO_ROOT,
    cfg_dict=cfg_cl,
    cfg_path=PATCHED_CFG_CL,
)
assert (RUN_DIR_CL / "best.pt").exists(), (
    f"CycleLayer training did not produce best.pt. See {LOG_DIR_CL}/train.log"
)
print(f"\nCycleLayer checkpoint: {RUN_DIR_CL / 'best.pt'} "
      f"({(RUN_DIR_CL / 'best.pt').stat().st_size / 1e6:.1f} MB)")


# ── Train CNN baseline (reuses splits written above) ─────────────────────
LOG_DIR_CNN = RUN_DIR_CNN / "logs"
LOG_DIR_CNN.mkdir(exist_ok=True)

_stream_subprocess(
    [sys.executable, "scripts/train.py",
     "--config", PATCHED_CFG_CNN, "--model", "cnn"],
    LOG_DIR_CNN / "train.log",
    cwd=REPO_ROOT,
    cfg_dict=cfg_cnn,
    cfg_path=PATCHED_CFG_CNN,
)
assert (RUN_DIR_CNN / "best.pt").exists(), (
    f"CNN training did not produce best.pt. See {LOG_DIR_CNN}/train.log"
)
print(f"\nCNN checkpoint: {RUN_DIR_CNN / 'best.pt'} "
      f"({(RUN_DIR_CNN / 'best.pt').stat().st_size / 1e6:.1f} MB)")


In [ ]:

# --- Cell: 8 — Evaluate both models (dev + test) ---

def _evaluate(split: str, run_dir: Path, cfg_path: Path, log_dir: Path) -> dict:
    """Run evaluate.py for one split; return loaded metrics dict."""
    metrics_path = run_dir / f"metrics_{split}.json"
    _stream_subprocess(
        [sys.executable, "scripts/evaluate.py",
         "--config",     cfg_path,
         "--checkpoint", run_dir / "best.pt",
         "--split",      split,
         "--output",     metrics_path],
        log_dir / f"eval_{split}.log",
        cwd=REPO_ROOT,
    )
    default_pred = run_dir / "predictions.csv"
    split_pred   = run_dir / f"predictions_{split}.csv"
    if default_pred.exists() and not split_pred.exists():
        default_pred.rename(split_pred)
    return json.load(open(metrics_path)) if metrics_path.exists() else {}


# ── Evaluate CycleLayer ───────────────────────────────────────────────────
metrics_cl_dev  = _evaluate("dev",  RUN_DIR_CL,  PATCHED_CFG_CL,  LOG_DIR_CL)
metrics_cl_test = _evaluate("test", RUN_DIR_CL,  PATCHED_CFG_CL,  LOG_DIR_CL)

# ── Evaluate CNN ──────────────────────────────────────────────────────────
metrics_cnn_dev  = _evaluate("dev",  RUN_DIR_CNN, PATCHED_CFG_CNN, LOG_DIR_CNN)
metrics_cnn_test = _evaluate("test", RUN_DIR_CNN, PATCHED_CFG_CNN, LOG_DIR_CNN)

# ── Quick summary ─────────────────────────────────────────────────────────
print("\n" + "="*64)
print("EVALUATION SUMMARY")
print("="*64)
for model_label, m_dev, m_test in [
    ("CycleLayer v1", metrics_cl_dev,  metrics_cl_test),
    ("CNN baseline",  metrics_cnn_dev, metrics_cnn_test),
]:
    print(f"\n  {model_label}")
    for split_name, m in [("dev", m_dev), ("test", m_test)]:
        if m:
            print(
                f"    [{split_name}]  "
                f"RMSE={m.get('rmse', float('nan')):.4f}  "
                f"S/win={m.get('s_score_mean', float('nan')):.4f}  "
                f"PH={m.get('ph_median', 'n/a')}  "
                f"PH-none={m.get('ph_none_count', 'n/a')}"
            )


In [ ]:

# =============================================================================
# --- Cell: 9 — LOUO Cross-Validation  (Advanced / Optional) ---
# Leave-One-Unit-Out for the CycleLayer model only.
# Runs 6 folds, each training a full model — roughly 6× the wall-time of
# a single training run.
# Enable with: RUN_LOUO = True in Cell 2.
# =============================================================================

if RUN_LOUO:
    print(f"Running LOUO for cyclelayer_v1 ...")

    _louo_anchor = RUN_DIR_CL / "_louo_anchor"
    _cfg_louo = yaml.safe_load(open(PATCHED_CFG_CL))
    _cfg_louo["training"]["output_dir"] = str(_louo_anchor)

    _louo_cfg_path = RUNTIME_CFG_DIR / f"louo_cyclelayer_v1_stride{STRIDE_TRAIN}.yaml"
    with open(_louo_cfg_path, "w") as _f:
        yaml.dump(_cfg_louo, _f, default_flow_style=False, sort_keys=False)

    _stream_subprocess(
        [sys.executable, "scripts/run_louo.py",
         "--config", _louo_cfg_path,
         "--model",  "cyclelayer_v1"],
        RUN_DIR_CL / "logs" / "louo.log",
        cwd=REPO_ROOT,
    )

    # results.json is saved by run_louo.py at {parent(output_dir)}/louo_cyclelayer_v1/
    _results_json = _louo_anchor.parent / "louo_cyclelayer_v1" / "results.json"
    if _results_json.exists():
        _r = json.load(open(_results_json))
        print(f"\nLOUO Results — cyclelayer_v1")
        print(f"  RMSE    : {_r.get('rmse_mean',    float('nan')):.4f}"
              f" ± {_r.get('rmse_std',    float('nan')):.4f}")
        print(f"  S-score : {_r.get('s_score_mean', float('nan')):.2f}"
              f" ± {_r.get('s_score_std', float('nan')):.2f}")
        print(f"  PH med  : {_r.get('ph_median', 'NaN')}")
        print(f"  PH none : {_r.get('ph_none_count', '?')}/{_r.get('n_folds', '?')} folds")
    else:
        print(f"[WARN] results.json not found at {_results_json}")
else:
    print("LOUO skipped (RUN_LOUO=False in Cell 2).")


In [ ]:

# --- Cell: 8b — Diagnostic Plots (both models) ---
# Generates trajectory plots, error histograms, scatter, spike locator CSV,
# and a self-contained report.md for each model.

for _model_label, _run_dir, _log_dir in [
    ("CycleLayer v1", RUN_DIR_CL,  LOG_DIR_CL),
    ("CNN baseline",  RUN_DIR_CNN, LOG_DIR_CNN),
]:
    _analysis_dir = _run_dir / "analysis"
    _analysis_dir.mkdir(exist_ok=True)
    print(f"\n{'='*60}")
    print(f"Diagnostics: {_model_label}")
    print(f"{'='*60}")

    for _split in ["dev", "test"]:
        _csv = _run_dir / f"predictions_{_split}.csv"
        if not _csv.exists():
            print(f"  [SKIP] No predictions CSV for split={_split}")
            continue

        _split_out = _analysis_dir / _split
        _split_out.mkdir(exist_ok=True)

        _stream_subprocess(
            [sys.executable, "scripts/analyze_predictions.py",
             "--pred_csv", _csv,
             "--out_dir",  _split_out,
             "--n_spikes", "20"],
            logfile=_log_dir / f"analyze_{_split}.log",
            cwd=REPO_ROOT,
        )
        print(f"\n  Outputs [{_split}]:")
        for _f in sorted(_split_out.iterdir()):
            print(f"    {_f.name}  ({_f.stat().st_size / 1e3:.1f} KB)")

    # Show first dev trajectory inline (Colab)
    try:
        from IPython.display import Image, display
        _first = next((_analysis_dir / "dev").glob("unit_*_trajectory.png"), None)
        if _first:
            print(f"\n{_model_label} — dev trajectory ({_first.name}):")
            display(Image(str(_first)))
    except Exception:
        pass

print(f"\nDone.  Analysis dirs:")
print(f"  CycleLayer : {RUN_DIR_CL  / 'analysis'}")
print(f"  CNN        : {RUN_DIR_CNN / 'analysis'}")


In [ ]:

# --- Cell: 8c — Compare CycleLayer vs CNN baseline ---
# Runs scripts/compare_runs.py to produce a structured table and comparison.md.
# CNN is always run_a (reference); CycleLayer is always run_b (challenger).

_stream_subprocess(
    [
        sys.executable, "scripts/compare_runs.py",
        "--run_a",   RUN_DIR_CNN,
        "--run_b",   RUN_DIR_CL,
        "--label_a", "CNN",
        "--label_b", "CycleLayer",
        "--out_dir", COMPARISON_DIR,
        "--splits",  "dev", "test",
    ],
    logfile=COMPARISON_DIR / "compare.log",
    cwd=REPO_ROOT,
)

# Render comparison.md inline in Colab
_md_path = COMPARISON_DIR / "comparison.md"
if _md_path.exists():
    try:
        from IPython.display import Markdown, display
        display(Markdown(_md_path.read_text(encoding="utf-8")))
    except Exception:
        print(_md_path.read_text(encoding="utf-8"))

print(f"\nComparison saved : {_md_path}")
print(f"CNN run dir      : {RUN_DIR_CNN}")
print(f"CycleLayer run   : {RUN_DIR_CL}")


In [ ]:

# --- Cell: 10 — Artifact Inventory ---

def _fmt_size(path: Path) -> str:
    return f"{path.stat().st_size / 1e6:.2f} MB" if path.exists() else "(absent)"

_ARTIFACT_FILES = [
    "best.pt", "last.pt",
    "theta_scaler.npz", "sensor_scaler.npz",
    "metrics_dev.json", "metrics_test.json",
    "predictions_dev.csv", "predictions_test.csv",
    "logs/train.log", "logs/eval_dev.log", "logs/eval_test.log",
]

for _model_label, _run_dir in [("CycleLayer v1", RUN_DIR_CL), ("CNN baseline", RUN_DIR_CNN)]:
    print(f"\nArtifacts — {_model_label}  ({_run_dir})")
    print(f"  {'File':<40} {'Size':>10}")
    print("  " + "-" * 52)
    for _fname in _ARTIFACT_FILES:
        _p = _run_dir / _fname
        print(f"  {_fname:<40} {_fmt_size(_p):>10}")

print(f"\nComparison report: {COMPARISON_DIR / 'comparison.md'}")


In [ ]:
if DRIVE_RUNS_ROOT is not None:
    DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)

    # Create zip archive of the run directory
    zip_base = str(DRIVE_RUNS_ROOT / RUN_ID)
    print(f"Zipping {RUN_DIR} -> {zip_base}.zip ...")
    zip_path = shutil.make_archive(
        zip_base,
        format="zip",
        root_dir=str(RUN_DIR.parent),
        base_dir=str(RUN_DIR.name),
    )
    zip_size_mb = Path(zip_path).stat().st_size / 1e6
    print(f"Created: {zip_path}  ({zip_size_mb:.1f} MB)")

    # Also copy patched config alongside the zip for reference
    shutil.copy2(PATCHED_CFG, DRIVE_RUNS_ROOT / PATCHED_CFG.name)
    print(f"Config copy: {DRIVE_RUNS_ROOT / PATCHED_CFG.name}")
else:
    print("DRIVE_RUNS_ROOT is None — skipping Drive upload.")
    print("To save artifacts manually:")
    print(f"  from google.colab import files")
    print(f"  files.download('{RUN_DIR}/best.pt')")


# --- Cell: 12 — Optional: View TensorBoard ---

In [ ]:
# Uncomment and run to launch TensorBoard in Colab:

%load_ext tensorboard
%tensorboard --logdir /content/runs

# Or equivalently:
# _sh(f"tensorboard --logdir {RUNS_ROOT} --port 6006 &")
# from google.colab.output import eval_js
# print(eval_js("google.colab.kernel.proxyPort(6006)"))


# --- Cell: 13 — Troubleshooting Notes ---

In [ ]:

NOTES = textwrap.dedent("""
    TROUBLESHOOTING
    ===============

    1. OOM (CUDA out of memory)
       - The script automatically retries with batch_size={fallback} on OOM.
       - Manually: set BATCH_SIZE = 256 or 128 in Cell 2.
       - Last resort: add  cfg["data"]["window_size"] = 20  in Cell 6.

    2. HDF5 file not found
       - Verify DRIVE_DATA_PATH in Cell 2 matches the exact path in Google Drive.
       - Google Drive paths are case-sensitive on Colab.

    3. 'No module named scripts' or 'No module named cyclelayer.data'
       - evaluate.py and run_louo.py add REPO_ROOT to sys.path automatically.
       - If still failing, add  sys.path.insert(0, str(REPO_ROOT))  at the top
         of the failing script and re-run.

    4. PH = None for all units
       - Expected at high stride (stride >= 50) or with few epochs.
       - With stride=5, >=50 epochs, GPU: model typically achieves PH for
         some units.  Use Val/pearson_r in TensorBoard to diagnose.
         If pearson_r > 0.9 but PH=None -> mid-trajectory spike problem.
         If pearson_r < 0.5               -> bias/collapse problem.

    5. Slow training
       - Confirm use_amp=True is active (shows in train log).
       - Reduce NUM_WORKERS if Colab shows DataLoader worker errors.
       - Use STRIDE_TRAIN = 10 or 50 for quick smoke tests.

    6. LOUO very slow
       - LOUO runs 6 full training loops (~6x single-run time).
       - Narrow with  --units 1 3  in run_louo.py to run only specific folds.

    7. Reproducing a run
       - The frozen config is stored in the run dir and zipped to Drive.
       - Load it: yaml.safe_load(open("config_frozen.yaml"))
       - Theta scaler: np.load("theta_scaler.npz")  -> mean, std arrays

    TENSORBOARD PANELS
    ==================

    Launch: Cell 12 (%tensorboard --logdir /content/runs)
    All panels are under the SCALARS tab.

    Loss/
      train_epoch, val_epoch  -- Total loss curve. Val > Train for many
                                 epochs = overfitting; val spiking = instability.
      train_rul_epoch         -- RUL component of training loss (MSE+asym).
      train_theta_epoch       -- Theta supervision component (Huber).
                                 If this dominates early on, increase warmup.
      val_rul_epoch           -- Val RUL loss. The most important loss for
                                 monitoring PH and RMSE progress.
      {k}_step                -- Per-step losses for dense monitoring.

    Val/                      [PLAUSIBILITY - check these first]
      mae                     -- Mean absolute error on val set. Should decrease
                                 smoothly. Sudden jump = instability.
      rmse                    -- Root MSE. Should track with mae.
      bias                    -- mean(pred - target). Positive = over-estimating
                                 RUL (model predicts longer life than actual).
                                 Watch: positive bias -> large S-score.
      pearson_r               -- Correlation between pred and target.
                                 ~ 0.95+ = model tracks RUL trend well.
                                 < 0.5   = model is near-constant.
                                 Sudden drop = collapse or spike.
      pred_mean, pred_std     -- Output distribution. pred_std -> 0 = collapse.
                                 pred_mean should track target_mean.
      pred_min, pred_max      -- Range check. Should span [0, max_rul].
      target_mean             -- Constant reference (sanity check).

    Train/
      mae, bias               -- Same stats for training set (per-batch approx).
                                 Large Train/mae vs Val/mae gap = overfitting.

    Theta/                    [PHYSICS / HEALTH PARAMS - cyclelayer_v1 only]
      mae_mean                -- Mean MAE across all 10 health parameters.
                                 Should decrease during the ramp phase.
      mae_fan_eff etc.        -- Per-channel MAE. Identifies which health
                                 parameters are hard to predict.
                                 Note: values are in normalised (z-score) space.
      pred_mean_{name}        -- Mean predicted value per channel (normalised).
                                 Should converge toward true_mean_{name}.
      true_mean_{name}        -- Mean true value per channel (constant ref).

    GradNorm/
      train_epoch             -- Mean pre-clip gradient L2 norm per epoch.
                                 > 10 for many epochs = LR too high or
                                 lambda_theta too large too early.
                                 Steadily near clip_norm = clipping is active;
                                 consider lowering LR.
      train_step              -- Per-step norm (noisy; use epoch for trend).

    Weights/
      param_norm              -- L2 norm of all model parameters.
                                 Should grow slowly and stabilise.
                                 Sudden jump = weight explosion (lower LR).

    Lambda/theta              -- Scheduled lambda_theta value per epoch.
                                 With 'delayed' schedule: 0 for warmup epochs,
                                 then ramp to lambda_theta_end.

    LR                        -- Learning rate (cosine decay).

    WHAT TO LOOK FOR
    ================

    Healthy training:
      Val/mae decreasing -> Val/pearson_r rising -> GradNorm stable -> Weights/param_norm stable

    Early instability (best checkpoint at epoch 3-5):
      GradNorm/train_epoch >> 1.0  in early epochs
      Fix: lower lr (3e-4), use 'delayed' schedule, increase warmup to 20 epochs

    Model collapse (PH=None, flat predictions):
      Val/pred_std -> 0, Val/pearson_r -> 0
      Fix: lower LR, check theta normalisation

    Over-estimation bias (large S-score):
      Val/bias > 0 persistently
      Fix: increase asymmetry weight in config, check asymmetry=0.5

    Theta supervision hurting RUL:
      Val/val_rul increasing while Val/theta_mae decreasing
      Fix: reduce lambda_theta_end or increase warmup_epochs
""").format(fallback=OOM_FALLBACK_BATCH)

print(NOTES)
